# Task 3, Step 6: Compute Allocation and Failure Cases

**Manual step 6.** Bring the evidence together:
- Accuracy against candidate count and generated tokens, separating oracle coverage from achieved selection accuracy.
- Whether additional samples add useful alternatives or repeat existing answers.
- Adaptive policies against fixed budgets at similar observed costs.
- Stopping behaviour against answer agreement and the fraction of correct candidates, used only retrospectively.
- Representative cases of selection disagreement and premature stopping, chosen by a stated rule.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task3_test_time_compute")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task3_test_time_compute.src.notebook_utils import *
setup_notebook()

In [ ]:
from task3_test_time_compute.src.evaluate import make_orders, select_for_prefix
questions = {d: split_questions(d)[1] for d in DATASETS}
offline = {d: pd.read_csv(f"{RESULTS_DIR}/offline_{d}.csv") for d in DATASETS}
coverage = {d: pd.read_csv(f"{RESULTS_DIR}/coverage_{d}.csv") for d in DATASETS}
best_method = {"math": "self_certainty_rwc", "mathvista": "visconf_rwc"}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    oracle = coverage[dataset].groupby("N")["oracle_coverage"].mean() * 100
    sc = offline[dataset][offline[dataset]["method"] == "self_consistency"].groupby("N")["accuracy"].mean() * 100
    quality = offline[dataset][offline[dataset]["method"] == best_method[dataset]].groupby("N")["accuracy"].mean() * 100
    first = offline[dataset][offline[dataset]["method"] == "first_ans"].groupby("N")["accuracy"].mean() * 100
    ax.fill_between(oracle.index, sc, oracle, alpha=0.18, color="#e67e22", label="Selection gap to oracle")
    ax.plot(oracle.index, oracle, marker="o", color="black", label="Oracle coverage")
    ax.plot(sc.index, sc, marker="o", label="Self-consistency")
    ax.plot(quality.index, quality, marker="o", label=METHOD_LABELS[best_method[dataset]])
    ax.plot(first.index, first, marker="o", linestyle="--", color="gray", label="FirstAns")
    ax.set(title=label, xlabel="Candidates N (log scale)", ylabel="% of questions", xscale="log")
    ax.set_xticks([1, 2, 4, 8, 16, 32]); ax.set_xticklabels([1, 2, 4, 8, 16, 32]); ax.legend(fontsize=8)
save_figure("task3_6_selection_gap")

## 6.1 What does each doubling buy?
Each doubling of candidates costs about the same factor in tokens. The bars show the extra accuracy and coverage it returns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
gain_tables = []
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    oracle = coverage[dataset].groupby("N")["oracle_coverage"].mean() * 100
    sc = offline[dataset][offline[dataset]["method"] == "self_consistency"].groupby("N")["accuracy"].mean() * 100
    gains = pd.DataFrame({"Oracle coverage": oracle.diff(), "Self-consistency accuracy": sc.diff()}).dropna()
    gains.index = [f"{int(n / 2)}→{int(n)}" for n in gains.index]
    gains.plot(kind="bar", ax=ax, edgecolor="black", rot=0)
    ax.axhline(0, color="black")
    ax.set(title=label, xlabel="Doubling of candidates", ylabel="Gain (percentage points)")
    gain_tables.append(gains.assign(dataset=dataset))
pd.concat(gain_tables).to_csv(f"{RESULTS_DIR}/marginal_gains.csv")
save_figure("task3_6_marginal_gains")

## 6.2 Do additional samples find new answers, and are they useful?
As candidates are revealed, a rollout is a **repeat** of a seen group, a **new wrong** group, or a **new correct** group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3), sharey=True)
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    test = questions[dataset]
    orders = make_orders(test)
    counts = np.zeros((32, 3))
    for q in test:
        for order in orders[q.sample_id]:
            seen = set()
            for position, rollout in enumerate(order):
                group = q.groups[rollout]
                kind = 0 if group in seen else (2 if q.group_correct[group] else 1)
                counts[position, kind] += 1
                seen.add(group)
    shares = counts / counts.sum(axis=1, keepdims=True) * 100
    ax.stackplot(np.arange(1, 33), shares.T, labels=["Repeat", "New wrong answer", "New correct answer"], colors=["#bdc3c7", "#e74c3c", "#2ecc71"])
    ax.set(title=label, xlabel="Position in reveal order", ylabel="% of rollouts", xlim=(1, 32), ylim=(0, 100)); ax.legend(loc="lower right", fontsize=9)
save_figure("task3_6_new_answers")

## 6.3 Stopping against agreement and correct-candidate fraction
The fraction of correct candidates in the full pool is used only here, retrospectively. No controller saw it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4))
for ax, (dataset, label) in zip(axes, DATASETS.items()):
    outcome = pd.read_csv(f"{RESULTS_DIR}/adaptive_outcomes_{dataset}_quality_aware.csv", dtype={"sample_id": str})
    per_q = outcome.groupby("sample_id")[["rollouts", "correct"]].mean()
    stats = pd.DataFrame([{"sample_id": q.sample_id, "correct_fraction": q.correct.mean(), "agreement": np.bincount(q.groups).max() / 32} for q in questions[dataset]]).set_index("sample_id").join(per_q)
    points = ax.scatter(stats["agreement"], stats["rollouts"] + np.random.default_rng(6304).uniform(-0.5, 0.5, len(stats)), c=stats["correct_fraction"], cmap="RdYlGn", s=14, alpha=0.75, vmin=0, vmax=1)
    plt.colorbar(points, ax=ax, label="Correct fraction of the 32-rollout pool")
    ax.set(title=f"{label}: quality-aware policy", xlabel="Share of largest answer group in the full pool", ylabel="Mean rollouts consumed")
save_figure("task3_6_stopping_vs_agreement")

## 6.4 Representative failure cases
**Selection rules, fixed before viewing.** Taken in held-out order at $N=32$ using the first saved order:
- **Selection disagreement:** first question where self-consistency and the quality-aware selector return different groups.
- **Premature stop:** first question and order where the quality-aware policy stopped early, was wrong, and fixed-32 was right.

In [ ]:
from task3_test_time_compute.src.evaluate import LLM_METHODS, VLM_METHODS
cases = []
for dataset, label in DATASETS.items():
    test = questions[dataset]
    orders = make_orders(test)
    for q in test:
        prefix = orders[q.sample_id][0]
        selected = select_for_prefix(q, prefix, VLM_METHODS if q.V is not None else LLM_METHODS)
        if selected["self_consistency"] != selected[best_method[dataset]]:
            cases.append({"Dataset": label, "Case": "Selection disagreement", "Question": q.sample_id, "Gold": q.gold[:50],
                          "Self-consistency": q.group_names[selected["self_consistency"]][:50], "Quality-aware": q.group_names[selected[best_method[dataset]]][:50],
                          "Votes per group": sorted(np.bincount(q.groups).tolist(), reverse=True)[:5]})
            break
    fixed = pd.read_csv(f"{RESULTS_DIR}/adaptive_outcomes_{dataset}_fixed_32.csv", dtype={"sample_id": str}).set_index(["sample_id", "order"])["correct"]
    outcome = pd.read_csv(f"{RESULTS_DIR}/adaptive_outcomes_{dataset}_quality_aware.csv", dtype={"sample_id": str}).set_index(["sample_id", "order"])
    harmful = outcome[(outcome["rollouts"] < 32) & (outcome["correct"] == 0) & (fixed.loc[outcome.index] == 1)]
    if len(harmful):
        sample_id, order = harmful.index[0]
        q = next(x for x in test if x.sample_id == sample_id)
        cases.append({"Dataset": label, "Case": f"Premature stop at {int(harmful.iloc[0]['rollouts'])}", "Question": sample_id, "Gold": q.gold[:50],
                      "Self-consistency": "fixed-32 was correct", "Quality-aware": "stopped early and was wrong", "Votes per group": sorted(np.bincount(q.groups).tolist(), reverse=True)[:5]})
cases = pd.DataFrame(cases)
cases.to_csv(f"{RESULTS_DIR}/representative_cases.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(cases)